## Final Proj Phase 1
### Lang Shao, Jiada Li

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

df = gpd.read_file('nsi_2022_25.gpkg')
df = df.dropna(subset=['firmzone'])

# as concept prooving code
# wesample only 10000 rows for fast training
sample_size = min(10000, len(df))
df = df.sample(n=sample_size, random_state=33)

# we set binary target for 1: High Risk, 0: Low Risk, 
# and instead of 'AREA', all the other not NaN values represented high risk
high_risk_zones = ['VE', 'AE', 'A', 'AO', 'AH']
df['target_label'] = df['firmzone'].isin(high_risk_zones).astype(int)
num_cols = ['ground_elv', 'found_ht', 'val_struct', 'med_yr_blt', 'sqft', 'num_story']
cat_cols = ['bldgtype', 'found_type']
if 'geometry' in df.columns:
    df = df.drop(columns=['geometry'])

/Users/samuelsiu/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# encode, split
Phi = pd.get_dummies(df[num_cols + cat_cols], drop_first=True)
x = Phi.values.astype(float)
y = df['target_label'].values.reshape(-1, 1)
split_idx = int(0.8 * len(x))
X_train_raw, X_test_raw = x[:split_idx], x[split_idx:]
Y_train, Y_test = y[:split_idx], y[split_idx:]

# normalze
X_mean = np.mean(X_train_raw, axis=0)
X_std = np.std(X_train_raw, axis=0)

X_train = (X_train_raw - X_mean) / (X_std + 1e-8)
X_test = (X_test_raw - X_mean) / (X_std + 1e-8)

In [3]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

In [4]:
# set up params
inp_node = X_train.shape[1]
hid_node = 8
out_node = 1
lr = 0.01
epochs = 1000

np.random.seed(33)
W1 = np.random.uniform(-1, 1, (inp_node, hid_node))
b1 = np.zeros((1, hid_node))
W2 = np.random.uniform(-1, 1, (hid_node, out_node))
b2 = np.zeros((1, out_node))
m = X_train.shape[0] 

for epoch in range(epochs):
    h_inp = np.dot(X_train, W1) + b1
    h1 = sigmoid(h_inp)
    h2 = np.dot(h1, W2) + b2
    pred_output = sigmoid(h2)
    
    error = Y_train - pred_output
    d_pred = error * sigmoid_derivative(pred_output)
    error_hid = d_pred.dot(W2.T)
    d_hid = error_hid * sigmoid_derivative(h1)
    dW2 = h1.T.dot(d_pred) / m
    db2 = np.sum(d_pred, axis=0, keepdims=True) / m
    dW1 = X_train.T.dot(d_hid) / m
    db1 = np.sum(d_hid, axis=0, keepdims=True) / m
    
    W2 += dW2 * lr
    b2 += db2 * lr
    W1 += dW1 * lr
    b1 += db1 * lr

    if epoch % 100 == 0:
        loss = np.mean(np.square(error))
        print(f"Epoch {epoch} | Loss: {loss:.4f}")

Epoch 0 | Loss: 0.4024
Epoch 100 | Loss: 0.3857
Epoch 200 | Loss: 0.3674
Epoch 300 | Loss: 0.3478
Epoch 400 | Loss: 0.3277
Epoch 500 | Loss: 0.3079
Epoch 600 | Loss: 0.2894
Epoch 700 | Loss: 0.2729
Epoch 800 | Loss: 0.2588
Epoch 900 | Loss: 0.2472


In [5]:
h_inp_test = np.dot(X_test, W1) + b1
h_out_test = sigmoid(h_inp_test)
out_input_test = np.dot(h_out_test, W2) + b2
pred_output_test = sigmoid(out_input_test)
predictions = (pred_output_test > 0.5).astype(int)
accuracy = np.mean(predictions == Y_test)

print(f"test acuracy: {accuracy * 100:.2f}%")

test acuracy: 58.60%
